# Task4 — Original + denoised inference

**ไม่เทรนใหม่** ใช้ EfficientNet fold models เดิม. กรอง Gaussian radius0.6 ที่ภาพต้นฉบับก่อน resize/pad/TTA. ทดสอบ baseline และ ResNetสองตัวตัวละ25% + EfficientNetภาพเดิม25% + EfficientNetภาพกรอง25%

สมมติฐาน: smoothing ลด high-frequency noise แต่ทำรายละเอียดเล็กหายได้ จึงเก็บ branch ภาพเดิมและเทียบกับ baseline. ไม่มีการอ้างว่ากรองแล้วดีกว่าแน่ ๆ

Run All ประเมินสอง foldsแล้วหยุด. โหลด baseline cache เดิม; inference ใหม่เฉพาะ branchกรองบน Train validation ภาพเดิมและ corruption pixels ชุดเดิม. ไม่ใช้ Test ไม่ปรับ sigma/weights ตามTest ไม่ทำsubmission

ค่าตัวกรอง/น้ำหนักกำหนดก่อนรัน ไม่ค้นหลายค่า. OOF/suiteถูกใช้ซ้ำจึง exploratory. แม้ไม่มีtraining ยังต้องอ่านภาพและรัน5-view inference หลายconditions จึงไม่ใช่งานทันที; cacheรายconditionช่วย resume

หากผ่านสองfoldต้องยืนยันอีกสามfoldและauditก่อนใช้จริง

In [1]:
from pathlib import Path
import sys, json, hashlib, time
import numpy as np
import pandas as pd
import torch
from dataclasses import asdict
from IPython.display import display
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'Task4/models/image_training.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from Task4.models import image_diversity_training as training
from Task4.models import image_denoise_evaluation as evaluation
from Task4.models import image_compound_validation as validation
from Task4.models.image_audit import make_folds
from Task4.models.image_compound_transforms import CONDITIONS
RAW=ROOT/'Task4/data/raw'
AUDIT=ROOT/'Task4/data/audit/followup_v2'
OUT=ROOT/'Task4/models/results/denoise_pilot_v1'
BATCH_SIZE=32
NUM_WORKERS=0
print('Device:',training.get_device())
if training.get_device().type=='cpu':raise RuntimeError('CPU training would be slow: select the project kernel with MPS/CUDA available.')


Device: mps


## 1. ตรวจ Train, folds และ checkpoint provenance

In [2]:
train=pd.read_csv(RAW/'train.csv')
groups=pd.read_parquet(AUDIT/'train_groups.parquet')
assert train.id.is_unique and train.label.notna().all()
assert all(np.array_equal(train[c],groups[c]) for c in ['id','file_name','label'])
metadata=pd.read_parquet(AUDIT/'images.parquet',filters=[('split','=','train')]).set_index('id').loc[train.id]
assert metadata.split.eq('train').all()
inventory=hashlib.sha256()
for row in metadata.itertuples():
    assert Path(row.file_name).name==row.file_name
    digest=validation.sha(RAW/'train'/row.file_name)
    assert digest==row.sha256, f'Changed Train image: {row.file_name}'
    inventory.update(f'{row.file_name}/{digest}'.encode())
folds=make_folds(groups,mode='near',n_splits=5,seed=42)
fold_hash=hashlib.sha256(b''.join(v.astype('int64').tobytes() for _,v in folds)).hexdigest()
assert fold_hash==json.loads((ROOT/'Task4/models/results/heavy_balance_v1/settings.json').read_text())['fold_hash']
reference_hashes={}
for fold,(fit,valid) in enumerate(folds,1):
    assert not np.intersect1d(fit,valid).size
    assert not set(groups.iloc[fit].near_group)&set(groups.iloc[valid].near_group)
    for path in evaluation.references(ROOT,fold):
        signature=json.loads((path.parent/'signature.json').read_text())
        assert signature['data_signature']==validation.sha_frame(groups.iloc[fit])
        reference_hashes[str(path.relative_to(ROOT))]=validation.sha(path)
        frame=pd.read_parquet(path.parent/'oof.parquet')
        assert frame.id.tolist()==groups.iloc[valid].id.tolist()
        assert np.array_equal(frame.true,groups.iloc[valid].label)
        reference_hashes[str((path.parent/'oof.parquet').relative_to(ROOT))]=validation.sha(path.parent/'oof.parquet')
sources={p.name:validation.sha(p) for p in (ROOT/'Task4/models').glob('image_*.py')}
settings={'recipe':{'filter':'GaussianBlur','radius':0.6,'blend_weights':[.25,.25,.25,.25],'folds':[1,2]},'batch_size':BATCH_SIZE,'workers':NUM_WORKERS,
          'train_manifest':validation.sha(RAW/'train.csv'),'inventory':inventory.hexdigest(),'fold_hash':fold_hash,
          'references':reference_hashes,'sources':sources,'torch':str(torch.__version__),'device':str(training.get_device())}
OUT.mkdir(parents=True,exist_ok=True)
if (OUT/'settings.json').exists():assert json.loads((OUT/'settings.json').read_text())==settings,'Settings changed: use a new OUT'
else:
    assert not any(OUT.iterdir()),'Nonempty OUT without settings'
    validation.write_json(OUT/'settings.json',settings)
print('Train verified:',len(train),'rows; Test unopened')

Train verified: 31546 rows; Test unopened


## 2. ประเมินสอง folds — ไม่ฝึกโมเดล

In [3]:
parts=[]
for fold in (1,2):
    started=time.monotonic()
    score=evaluation.compare(ROOT,RAW,groups,folds,OUT,fold)
    parts.append(score)
    display(evaluation.summarize(score))
    print('Fold',fold,'evaluation minutes:',round((time.monotonic()-started)/60,1))
combined=pd.concat(parts,ignore_index=True)
combined.to_parquet(OUT/'two_fold_scores.parquet',index=False)
display(evaluation.summarize(combined))
s=combined.copy()
s['family']=s.condition.map(lambda c:'original' if c=='original' else ('compound' if c.startswith('compound_') else CONDITIONS[c][1]))
f=s.groupby(['procedure','fold','family']).macro_F1.mean()
delta=(f.loc['denoise_blend25']-f.loc['baseline']).unstack('family')
report=delta[['original']].assign(stress_family_gain=delta.drop(columns='original').mean(axis=1))
display(report)
validation.write_json(OUT/'two_fold_report.json',{'gains':report.reset_index().to_dict('records'),'test_used':False,'training_performed':False})
print('STOP: two-fold screening complete. No Test inference/submission.')

stress original original
stress stress gaussian_1
stress stress gaussian_2
stress stress speckle_1
stress stress speckle_2
stress stress blur_1
stress stress blur_2
stress stress jpeg_1
stress stress jpeg_2
stress stress dark_1
stress stress dark_2
stress stress compound_dark_noise
stress stress compound_color_noise
stress stress compound_rotation_dark
stress stress compound_bright_color
stress stress compound_dark_color_noise
stress stress compound_rotation_color_noise


suite,compound,original,single
procedure,,,
baseline,0.944725,0.986331,0.977201
denoise_blend25,0.946052,0.986241,0.977799


Fold 1 evaluation minutes: 3.7
stress original original
stress stress gaussian_1
stress stress gaussian_2
stress stress speckle_1
stress stress speckle_2
stress stress blur_1
stress stress blur_2
stress stress jpeg_1
stress stress jpeg_2
stress stress dark_1
stress stress dark_2
stress stress compound_dark_noise
stress stress compound_color_noise
stress stress compound_rotation_dark
stress stress compound_bright_color
stress stress compound_dark_color_noise
stress stress compound_rotation_color_noise


suite,compound,original,single
procedure,,,
baseline,0.955820,0.983375,0.9856
denoise_blend25,0.956457,0.983718,0.9858


Fold 2 evaluation minutes: 3.7


suite,compound,original,single
procedure,,,
baseline,0.950273,0.984853,0.981401
denoise_blend25,0.951254,0.984980,0.981799


family,original,stress_family_gain
fold,,
1,-0.000090,0.000719
2,0.000343,0.000273


STOP: two-fold screening complete. No Test inference/submission.
